In [ ]:
print("Hello, World!")

In [2]:
import os

from dotenv import load_dotenv

from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.runnables import RunnablePassthrough
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma

In [3]:
import sys
print(sys.executable)

c:\Users\roger\Documents\SabioGroup\Final Projects\.venv\Scripts\python.exe


In [4]:
# Load environment variables
load_dotenv()

True

In [7]:
# ==========================================
# 1. LOAD THE PDF
# ==========================================

loader = PyPDFLoader("mg_service_manual.pdf")

car_docs = loader.load()

print(f"Pages loaded: {len(car_docs)}")



Pages loaded: 5


In [8]:
# ==========================================
# 2. LOAD THE LLM
# ==========================================

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

In [9]:
# ==========================================
# 3. LOAD THE EMBEDDING MODEL
# ==========================================

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

In [12]:
# ==========================================
# 4. SPLIT THE PDF INTO CHUNKS
# ==========================================

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=50
)

splits = text_splitter.split_documents(car_docs)

print(f"Chunks created: {len(splits)}")


Chunks created: 61


In [13]:
# ==========================================
# 5. CREATE CHROMA VECTOR DATABASE
# ==========================================

vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embeddings
)


In [14]:
# ==========================================
# 6. CREATE THE RETRIEVER
# ==========================================

retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)


In [15]:

# ==========================================
# 7. CREATE THE RAG PROMPT
# ==========================================

prompt = ChatPromptTemplate.from_template("""
You are a helpful car assistant.

Use the following context from the car manual to answer
the user's question.

If you don't know the answer based on the context,
say that you don't know.

Keep your answer concise and use a maximum of
three sentences.

Context:
{context}

Question:
{question}

Answer:
""")


In [16]:
# ==========================================
# 8. CREATE THE RAG CHAIN
# ==========================================

rag_chain = (
    {
        "context": retriever,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
)



In [21]:
# ==========================================
# 9. ASK A QUESTION
# ==========================================

# query = """
# The Gasoline Particulate Filter Full warning has appeared.
# What does this mean and what should I do?
# """

query = """
    there is red light blinking on the dashboard and the car is making a beeping sound. What does this mean and what should I do?
"""


In [22]:
# ==========================================
# 10. GET THE ANSWER
# ==========================================

answer = rag_chain.invoke(query).content

print("\nAnswer:")
print(answer)


Answer:
A blinking red light on the dashboard, accompanied by a beeping sound, typically indicates a critical issue that requires immediate attention. You should not continue driving and should consult the vehicle manual or arrange for a professional diagnosis.
